In [1]:
#!pip install geospark

import json

from collections import defaultdict, namedtuple
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.statcounter import StatCounter
from pyspark.rdd import RDD

from shapely.geometry import Point, shape

In [2]:
# Initialize SparkSession & SparkContext
spark = SparkSession.builder \
    .appName("RunTaxiTrips") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Define TaxiTrip class as a 'namedtuple'
TaxiTrip = namedtuple('TaxiTrip', ['pickupTime', 'dropoffTime', 'pickupLoc', 'dropoffLoc'])

def parse(line: str):
    fields = line.split(',')
    license = fields[1]
    pickupTime = datetime.strptime(fields[5], "%Y-%m-%d %H:%M:%S")
    dropoffTime = datetime.strptime(fields[6], "%Y-%m-%d %H:%M:%S")
    pickupLoc = (float(fields[10]), float(fields[11]))
    dropoffLoc = (float(fields[12]), float(fields[13]))
    trip = TaxiTrip(pickupTime, dropoffTime, pickupLoc, dropoffLoc)
    return (license, trip)

def safe(f):
    def wrapper(s):
        try:
            return f(s)
        except Exception as e:
            return (s, e)
    return wrapper

In [4]:
# Parse & Filter the Taxi Trips

taxiRaw = spark.read.text("../Data/nyc-taxi-trips") #.sample(False, 0.01) # 1 percent sample size!

taxiParsed = taxiRaw.rdd.map(lambda row: safe(lambda x: parse(x))(row.value))
taxiParsed.cache()

# bad records
taxiBad = taxiParsed.filter(lambda x: isinstance(x[1], Exception))

# good records 
taxiGood = taxiParsed.filter(lambda x: not isinstance(x[1], Exception))
taxiGood.cache()  # cache good lines for later re-use

print("Number of bad trips:", taxiBad.count())
print("Number of good trips:", taxiGood.count())

for trip in taxiGood.take(5):
    print()
    print(trip)
    print("pickupTime:", trip[1].pickupTime)
    print("dropoffTime:", trip[1].dropoffTime)
    print("pickupLoc:", trip[1].pickupLoc)
    print("dropoffLoc:", trip[1].dropoffLoc)

Number of bad trips: 87


Number of good trips: 14776529

('BA96DE419E711691B9445D6A6307C170', TaxiTrip(pickupTime=datetime.datetime(2013, 1, 1, 15, 11, 48), dropoffTime=datetime.datetime(2013, 1, 1, 15, 18, 10), pickupLoc=(-73.978165, 40.757977), dropoffLoc=(-73.989838, 40.751171)))
pickupTime: 2013-01-01 15:11:48
dropoffTime: 2013-01-01 15:18:10
pickupLoc: (-73.978165, 40.757977)
dropoffLoc: (-73.989838, 40.751171)

('9FD8F69F0804BDB5549F40E9DA1BE472', TaxiTrip(pickupTime=datetime.datetime(2013, 1, 6, 0, 18, 35), dropoffTime=datetime.datetime(2013, 1, 6, 0, 22, 54), pickupLoc=(-74.006683, 40.731781), dropoffLoc=(-73.994499, 40.75066)))
pickupTime: 2013-01-06 00:18:35
dropoffTime: 2013-01-06 00:22:54
pickupLoc: (-74.006683, 40.731781)
dropoffLoc: (-73.994499, 40.75066)

('9FD8F69F0804BDB5549F40E9DA1BE472', TaxiTrip(pickupTime=datetime.datetime(2013, 1, 5, 18, 49, 41), dropoffTime=datetime.datetime(2013, 1, 5, 18, 54, 23), pickupLoc=(-74.004707, 40.73777), dropoffLoc=(-74.009834, 40.726002)))
pickupTime: 2013-0

25/05/15 11:55:53 WARN BlockManager: Task 40 already completed, not releasing lock for rdd_6_0


In [5]:
taxiGood = taxiGood.filter(lambda x: x[1].pickupTime.day == 13) # instead of sampling, we filter by day (which does not disrupt the sessions)
taxiGood.cache()

print("Number of filtered trips:", taxiGood.count())

[Stage 3:======================================================>  (19 + 1) / 20]

Number of filtered trips: 442540


In [6]:
# Helper functions for trip durations and pickup/drop-off locations

def getHours(trip):
    pickup_time = trip.pickupTime
    dropoff_time = trip.dropoffTime
    duration_hours = (dropoff_time - pickup_time).total_seconds() / 3600
    return int(duration_hours)

# Select taxi trips based on trip duration!
taxiClean_rdd = taxiGood.filter(lambda row: 0 <= getHours(row[1]) < 3)

def is_valid_trip_location(row): # check for valid trip locations
    zero = (0.0, 0.0)
    pickup_loc = row[1].pickupLoc
    dropoff_loc = row[1].dropoffLoc
    return pickup_loc != zero and dropoff_loc != zero

# Remove trips with invalid pickup or drop-off locations
taxiDone_rdd = taxiClean_rdd.filter(is_valid_trip_location)
taxiDone_rdd.cache()

PythonRDD[12] at RDD at PythonRDD.scala:53

In [7]:
# Analyze the trip durations

duration_rdd = taxiGood.map(lambda x: (getHours(x[1]),))
duration_counts = duration_rdd.countByValue() # count the occurrences of each duration

print("Distribution of trip durations:")
for duration, count in duration_counts.items():
    print(f"Duration: < {duration[0]+1} hour(s) - Count: {count}")

Distribution of trip durations:
Duration: < 1 hour(s) - Count: 442209
Duration: < 2 hour(s) - Count: 289
Duration: < 3 hour(s) - Count: 27
Duration: < 4 hour(s) - Count: 6
Duration: < 5 hour(s) - Count: 3
Duration: < 8 hour(s) - Count: 3
Duration: < 6 hour(s) - Count: 2
Duration: < 12 hour(s) - Count: 1


In [8]:
# Parse the NYC Boroughs Polygons

with open("../Data/nyc-borough-boundaries-polygon.geojson", "r") as file:
    geojson = json.load(file)

features = geojson["features"]
bFeatures = spark.sparkContext.broadcast(features) # broadcast to all worker nodes

def borough(trip):
    dropoff_loc = (trip.pickupLoc[0], trip.pickupLoc[1])  # pickup location coordinates correctly
    p = Point(dropoff_loc)
    for feature in bFeatures.value:
        if feature["geometry"]["type"] == "Polygon":
            if p.within(shape(feature["geometry"])):
                return feature["properties"]["borough"]
    return None

In [9]:
borough_rdd = taxiDone_rdd.map(lambda x: borough(x[1]))
borough_counts = borough_rdd.countByValue()

print("Distribution of trips per borough:")
for borough, count in borough_counts.items():
    print(f"{borough}: {count}")

[Stage 5:======================================================>  (19 + 1) / 20]

Distribution of trips per borough:
Manhattan: 392392
Brooklyn: 16399
Queens: 24048
Bronx: 238
None: 838
Staten Island: 12


In [10]:
# Helper functions for "sessionization"

def secondaryKey(trip):
    return trip.pickupTime.timestamp()

def split(t1, t2):
    p1 = t1.pickupTime
    p2 = t2.pickupTime
    d = p2 - p1
    return d.total_seconds() / 3600 >= 4

def groupSorted(it, splitFunc):
    res = defaultdict(list)
    for (key, value) in it:
        res[key[0]].append(value)
    for key, value in res.items():
        yield (key, value)

def groupByKeyAndSortValues(rdd, secondaryKeyFunc, splitFunc, numPartitions):
    presess = rdd.map(lambda x: ((x[0], secondaryKeyFunc(x[1])), x[1]))
    return presess.partitionBy(numPartitions).sortByKey().mapPartitions(lambda x: groupSorted(x, splitFunc))

sessions = groupByKeyAndSortValues(taxiDone_rdd, secondaryKey, split, 30)  # use fixed amount of 30 partitions
sessions.cache()  

print("Sample sessions:\n")
for session in sessions.take(5):
    print(session, '\n')

Sample sessions:



[Stage 11:=============================>                         (16 + 14) / 30]

('0002555BBE359440D6CEB34B699D3932', [TaxiTrip(pickupTime=datetime.datetime(2013, 1, 13, 0, 12, 34), dropoffTime=datetime.datetime(2013, 1, 13, 0, 22, 51), pickupLoc=(-73.985199, 40.742062), dropoffLoc=(-74.00679, 40.744064)), TaxiTrip(pickupTime=datetime.datetime(2013, 1, 13, 0, 32, 27), dropoffTime=datetime.datetime(2013, 1, 13, 0, 43, 53), pickupLoc=(-74.005127, 40.741093), dropoffLoc=(-73.990349, 40.760841)), TaxiTrip(pickupTime=datetime.datetime(2013, 1, 13, 0, 54, 13), dropoffTime=datetime.datetime(2013, 1, 13, 0, 58, 2), pickupLoc=(-73.975624, 40.760941), dropoffLoc=(-73.987862, 40.747459)), TaxiTrip(pickupTime=datetime.datetime(2013, 1, 13, 1, 0, 27), dropoffTime=datetime.datetime(2013, 1, 13, 1, 22, 51), pickupLoc=(-73.989059, 40.747768), dropoffLoc=(-73.992088, 40.699482)), TaxiTrip(pickupTime=datetime.datetime(2013, 1, 13, 1, 28, 59), dropoffTime=datetime.datetime(2013, 1, 13, 1, 37, 43), pickupLoc=(-73.992409, 40.698929), dropoffLoc=(-73.94677, 40.708763)), TaxiTrip(pickupT

25/05/15 12:17:34 WARN BlockManager: Task 211 already completed, not releasing lock for rdd_25_0
                                                                                

In [11]:
# Final analysis of average wait-times per borough

def borough(trip):
    dropoff_loc = (trip.pickupLoc[0], trip.pickupLoc[1])  # Access pickup location coordinates correctly
    p = Point(dropoff_loc)
    for feature in bFeatures.value:
        if feature["geometry"]["type"] == "Polygon":
            if p.within(shape(feature["geometry"])):
                return feature["properties"]["borough"]
    return None

def boroughDuration(t1, t2):
    b = borough(t1)
    d = (t2.pickupTime - t1.dropoffTime).total_seconds()
    return (b, d)

boroughDurations = sessions.values().flatMap(lambda trips: (
    (boroughDuration(trips[i], trips[i + 1]) for i in range(len(trips) - 1))
)).cache()

print("Distribution of wait-times in hours:")
wait_times_hours = boroughDurations.values().map(lambda x: int(x / 3600)).countByValue()
for hours, count in sorted(wait_times_hours.items(), key=lambda x: x[0]):
    print(hours, count)

Distribution of wait-times in hours:


[Stage 15:======================================================> (29 + 1) / 30]

0 393005
1 9959
2 2179
3 659
4 329
5 160
6 102
7 77
8 105
9 174
10 273
11 506
12 1320
13 1488
14 844
15 466
16 236
17 120
18 64
19 29
20 18
21 7
22 5
23 3


In [12]:
print("Final stats of wait-times (in seconds) per borough:")
wait_times_per_borough = boroughDurations.filter(lambda x: x[1] >= 0).mapValues(
    lambda d: StatCounter().merge(d)
).reduceByKey(lambda a, b: a.mergeStats(b)).collect()
for borough, stats in wait_times_per_borough:
    print(borough, stats)

Final stats of wait-times (in seconds) per borough:
None (count: 758, mean: 3170.382585751979, stdev: 9625.050167230404, max: 65040.0, min: 0.0)
Queens (count: 21355, mean: 2285.4229922734726, stdev: 6203.326658777562, max: 78420.0, min: 0.0)
Bronx (count: 220, mean: 5004.322727272728, stdev: 12583.737927847838, max: 58620.0, min: 0.0)
Brooklyn (count: 15286, mean: 2299.890030092895, stdev: 8287.296409404082, max: 74781.0, min: 0.0)
Staten Island (count: 11, mean: 6839.181818181818, stdev: 16760.940946781226, max: 59580.0, min: 120.0)
Manhattan (count: 374409, mean: 1367.0984565007784, stdev: 5570.1145163637475, max: 84840.0, min: 0.0)
